In [20]:
import folium
from folium.plugins import MarkerCluster
import json
import pandas as pd
import html as html_module
import re
from math import radians, cos, sin, sqrt
from pathlib import Path
from IPython.display import display, HTML

# =========================================================
# CONFIG — chemins relatifs, rien à modifier
# =========================================================
BASE_DIR    = Path(".")
TEXTS_DIR   = BASE_DIR / "data_texts"
FR_PATH     = TEXTS_DIR / "20000lieues_fr.txt"
EN_PATH     = TEXTS_DIR / "20000lieues_an.txt"
JSON_PATH   = BASE_DIR / "escales_enrichies_dates_corrigees.json"
OUTPUT_HTML = BASE_DIR / "CIAVisualisationnautilusmultilang_fixed.html"

for p in [TEXTS_DIR, FR_PATH, EN_PATH, JSON_PATH]:
    assert p.exists(), f"❌ Introuvable : {p.resolve()}"
    print(f"✅ {p.resolve()}")

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
en_text_full = EN_PATH.read_text(encoding="utf-8", errors="ignore")
print(f"📖 FR : {len(fr_text_full):,} caractères")
print(f"📖 EN : {len(en_text_full):,} caractères")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"🗺️  {len(data)} escales chargées")

# =========================================================
# FONCTIONS UTILITAIRES
# =========================================================
def clean_text(txt):
    if txt is None:
        return ""
    txt = str(txt)
    txt = txt.replace("\u00a0", " ").replace("\ufeff", "")
    txt = re.sub(r"\r\n?", "\n", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

def get_lang(value, lang):
    """Lit la valeur FR ou EN depuis un champ bilingue du JSON."""
    if isinstance(value, dict):
        v = value.get(lang) or value.get("fr") or ""
        return clean_text(str(v)) if v else ""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return clean_text(str(value))

def get_passage(row, lang):
    """Lit passage_roman.fr ou passage_roman.en depuis le JSON v3."""
    passage = row.get("passage_roman", None)
    if isinstance(passage, dict):
        val = passage.get(lang)
        if val and str(val).strip() not in ("", "None", "null"):
            return clean_text(str(val))
        return ""
    if passage and isinstance(passage, str):
        return clean_text(passage)
    return ""

def distance_from_center(lat):
    R = 6371e3
    lat_rad = radians(lat)
    return int(sqrt((R * cos(lat_rad))**2 + (R * sin(lat_rad))**2))

def make_html_content(nom, texte, label_titre):
    """Formate un extrait en HTML pour la Bootstrap Modal."""
    paras = [p.strip() for p in texte.replace("\n\n", "\n").split("\n") if p.strip()]
    paras_html = "".join(f"<p style='margin-bottom:14px; line-height:1.8; font-size:1.2rem;'>{p}</p>" for p in paras[:4])
    return f"""
        <h3 style='color:#0a3d62; font-size:1.3rem; margin-bottom:8px;'>{nom}</h3>
        <h5 style='color:#27c39f; font-size:1rem; margin-bottom:16px; font-weight:600;'>{label_titre}</h5>
        {paras_html}"""

# =========================================================
# DATAFRAME
# =========================================================
df = pd.DataFrame(data)
df["Latitude_Décimal"]  = pd.to_numeric(df["Latitude_Décimal"],  errors="coerce")
df["Longitude_Décimal"] = pd.to_numeric(df["Longitude_Décimal"], errors="coerce")
df = df.dropna(subset=["Latitude_Décimal", "Longitude_Décimal"]).reset_index(drop=True)
df["DistanceCentreTerre"] = df["Latitude_Décimal"].apply(distance_from_center)
print(f"✅ {len(df)} escales avec coordonnées valides")

# =========================================================
# FICHE 2 — DICTIONNAIRE ET EXTRACTION DES ESPÈCES MARINES
# =========================================================

ESPECES_FR = {
    # Cétacés
    "baleine": "🐋 Baleine",
    "cachalot": "🐋 Cachalot",
    "dauphin": "🐬 Dauphin",
    "marsouin": "🐬 Marsouin",
    "narval": "🦷 Narval",
    "orque": "🐋 Orque",
    # Poissons
    "requin": "🦈 Requin",
    "raie": "🐟 Raie",
    "thon": "🐟 Thon",
    "saumon": "🐟 Saumon",
    "morue": "🐟 Morue",
    "hareng": "🐟 Hareng",
    "espadon": "🐟 Espadon",
    "murène": "🐟 Murène",
    "rémora": "🐟 Rémora",
    "lamproie": "🐟 Lamproie",
    "orphie": "🐟 Orphie",
    "dorade": "🐟 Dorade",
    "turbot": "🐟 Turbot",
    "sole": "🐟 Sole",
    "grondin": "🐟 Grondin",
    "baudroie": "🐟 Baudroie",
    "poisson volant": "🐟 Poisson volant",
    "poisson lune": "🐟 Poisson lune",
    # Mollusques et céphalopodes
    "poulpe": "🐙 Poulpe",
    "pieuvre": "🐙 Pieuvre",
    "calmar": "🦑 Calmar",
    "seiche": "🦑 Seiche",
    "nautile": "🐚 Nautile",
    "huître": "🦪 Huître",
    "moule": "🦪 Moule",
    "coquille": "🐚 Coquille",
    "pétoncle": "🦪 Pétoncle",
    "buccin": "🐚 Buccin",
    "triton": "🐚 Triton",
    "volute": "🐚 Volute",
    "cone": "🐚 Cône",
    "murex": "🐚 Murex",
    "tridacne": "🦪 Tridacne",
    "ptéropode": "🐚 Ptéropode",
    # Crustacés
    "homard": "🦞 Homard",
    "languste": "🦞 Langouste",
    "crabe": "🦀 Crabe",
    "crevette": "🦐 Crevette",
    "araignée de mer": "🦀 Araignée de mer",
    "bernard": "🦀 Bernard-l'ermite",
    "cirrhipède": "🦐 Cirrhipède",
    # Échinodermes
    "oursin": "🦔 Oursin",
    "étoile de mer": "⭐ Étoile de mer",
    "holothurie": "🐛 Holothurie",
    "astérie": "⭐ Astérie",
    "encrine": "🌿 Encrine",
    "crinoïde": "🌿 Crinoïde",
    "ophiure": "⭐ Ophiure",
    # Méduses et cnidaires
    "méduse": "🪼 Méduse",
    "physalie": "🪼 Physalie",
    "anémone": "🌸 Anémone de mer",
    "corail": "🪸 Corail",
    "gorgone": "🪸 Gorgone",
    "madrépore": "🪸 Madrépore",
    "alcyon": "🪸 Alcyon",
    "polype": "🪸 Polype",
    "zoophyte": "🌿 Zoophyte",
    # Algues et plantes marines
    "algue": "🌿 Algue",
    "varech": "🌿 Varech",
    "sargasse": "🌿 Sargasse",
    "fucus": "🌿 Fucus",
    "laminaire": "🌿 Laminaire",
    "posidonie": "🌿 Posidonie",
    # Mammifères marins
    "morse": "🦭 Morse",
    "phoque": "🦭 Phoque",
    "lamantin": "🐋 Lamantin",
    "dugong": "🐋 Dugong",
    "otarie": "🦭 Otarie",
    # Reptiles et oiseaux marins
    "tortue": "🐢 Tortue marine",
    "serpent de mer": "🐍 Serpent de mer",
    "albatros": "🐦 Albatros",
    "pingouin": "🐧 Pingouin",
    "manchot": "🐧 Manchot",
    "pétrel": "🐦 Pétrel",
    "mouette": "🐦 Mouette",
    # Autres
    "éponge": "🧽 Éponge",
    "thalassiophyte": "🌿 Thalassiophyte",
    "annélide": "🪱 Annélide",
    "vermicelle": "🪱 Ver marin",
    "hippocampe": "🐴 Hippocampe",
}

ESPECES_EN = {
    "whale": "🐋 Whale",
    "sperm whale": "🐋 Sperm whale",
    "cachalot": "🐋 Cachalot",
    "dolphin": "🐬 Dolphin",
    "porpoise": "🐬 Porpoise",
    "narwhal": "🦷 Narwhal",
    "shark": "🦈 Shark",
    "ray": "🐟 Ray",
    "tuna": "🐟 Tuna",
    "salmon": "🐟 Salmon",
    "cod": "🐟 Cod",
    "herring": "🐟 Herring",
    "swordfish": "🐟 Swordfish",
    "moray": "🐟 Moray eel",
    "flying fish": "🐟 Flying fish",
    "octopus": "🐙 Octopus",
    "squid": "🦑 Squid",
    "cuttlefish": "🦑 Cuttlefish",
    "nautilus": "🐚 Nautilus",
    "oyster": "🦪 Oyster",
    "mussel": "🦪 Mussel",
    "shell": "🐚 Shell",
    "lobster": "🦞 Lobster",
    "crab": "🦀 Crab",
    "shrimp": "🦐 Shrimp",
    "sea spider": "🦀 Sea spider",
    "sea urchin": "🦔 Sea urchin",
    "starfish": "⭐ Starfish",
    "sea cucumber": "🐛 Sea cucumber",
    "jellyfish": "🪼 Jellyfish",
    "physalia": "🪼 Physalia",
    "anemone": "🌸 Sea anemone",
    "coral": "🪸 Coral",
    "polyp": "🪸 Polyp",
    "seaweed": "🌿 Seaweed",
    "kelp": "🌿 Kelp",
    "sargasso": "🌿 Sargasso",
    "seal": "🦭 Seal",
    "walrus": "🦭 Walrus",
    "manatee": "🐋 Manatee",
    "turtle": "🐢 Sea turtle",
    "albatross": "🐦 Albatross",
    "penguin": "🐧 Penguin",
    "petrel": "🐦 Petrel",
    "sponge": "🧽 Sponge",
    "sea horse": "🐴 Sea horse",
    "poulp": "🐙 Poulp",
}

def extraire_especes(passage, lang="fr"):
    """
    Cherche les espèces marines dans un passage du roman.
    Retourne du HTML formaté pour la Bootstrap Modal.
    """
    if not passage or "intermédiaire" in passage or "intermediate" in passage:
        if lang == "fr":
            return "<p>🗺️ Escale intermédiaire — aucun passage du roman associé.</p>"
        else:
            return "<p>🗺️ Intermediate stop — no novel passage associated.</p>"

    passage_lower = passage.lower()
    dico = ESPECES_FR if lang == "fr" else ESPECES_EN
    trouvees = []

    for mot, label in dico.items():
        if mot.lower() in passage_lower:
            trouvees.append(label)

    if lang == "fr":
        titre_escale_label = "🐙 Espèces maritimes mentionnées"
        msg_vide = "<p>Aucune espèce marine identifiée dans ce passage.</p><p><em>Jules Verne décrit ici plutôt la géographie ou les événements du voyage.</em></p>"
        intro = f"<p style='color:#607080; font-size:0.85rem; margin-bottom:14px;'>Espèces identifiées dans l'extrait du roman :</p>"
    else:
        titre_escale_label = "🐙 Marine species mentioned"
        msg_vide = "<p>No marine species identified in this passage.</p><p><em>Jules Verne describes geography or journey events here.</em></p>"
        intro = f"<p style='color:#607080; font-size:0.85rem; margin-bottom:14px;'>Species identified in the novel excerpt:</p>"

    if not trouvees:
        return msg_vide

    # Formatage en badges
    badges = "".join(
        f"<span style='"
        f"display:inline-block; margin:4px; padding:5px 12px;"
        f"background:rgba(39,195,159,0.12); color:#27c39f;"
        f"border:1px solid rgba(39,195,159,0.3); border-radius:20px;"
        f"font-size:0.9rem;'>{e}</span>"
        for e in trouvees
    )

    return f"{intro}<div style='display:flex; flex-wrap:wrap; gap:4px;'>{badges}</div>"

print(f"✅ Fiche 2 — {len(ESPECES_FR)} espèces FR · {len(ESPECES_EN)} espèces EN dans le dictionnaire")

# =========================================================
# CONTENUS DÉTAILLÉS — FR ET EN SÉPARÉS
# Même structure que l'original : DETAILED_CONTENT[stop][lang][action]
# Les valeurs sont du HTML prêt à injecter dans la Bootstrap Modal
# =========================================================
MSG_INTER_FR = "<p>🗺️ Ce point est une escale de navigation intermédiaire, non décrite explicitement dans le roman de Jules Verne.</p>"
MSG_INTER_EN = "<p>🗺️ This is an intermediate navigation point, not explicitly described in Jules Verne's novel.</p>"

detailed_content = {}

for i, row in df.iterrows():
    stop_fr  = get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}"
    stop_en  = get_lang(row.get("Escales du Nautilus"), "en") or f"Stop {i+1}"
    event_fr = get_lang(row.get("Event"), "fr")
    event_en = get_lang(row.get("Event"), "en")

    passage_fr = get_passage(row, "fr")
    passage_en = get_passage(row, "en")

    # Points intermédiaires sans passage
    if not passage_fr and not passage_en:
        book_html_fr = f"<h3 style='color:#0a3d62;'>{stop_fr}</h3>{MSG_INTER_FR}"
        book_html_en = f"<h3 style='color:#0a3d62;'>{stop_en}</h3>{MSG_INTER_EN}"
    else:
        book_html_fr = make_html_content(
            stop_fr,
            passage_fr if passage_fr else "Extrait indisponible pour cette escale.",
            "📖 Extrait du livre"
        )
        book_html_en = make_html_content(
            stop_en,
            passage_en if passage_en else "Excerpt unavailable for this stop.",
            "📖 Book excerpt"
        )

    detailed_content[str(i)] = {
        # ── FRANÇAIS ──────────────────────────────────────────
        "fr": {
            "book_excerpt": book_html_fr,
            "marine_species": extraire_especes(passage_fr, "fr"),
            "science_19c": f"""
                <h3 style='color:#0a3d62;'>Découvertes scientifiques du XIXe siècle</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {stop_fr}</h5>
                <p><strong>Contexte de l'escale :</strong> {event_fr}</p>
                <p>Fiche 3 — en cours de développement.</p>""",
            "compare_1866_today": f"""
                <h3 style='color:#0a3d62;'>Comparaison entre 1866 et aujourd'hui</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>⚗️ {stop_fr}</h5>
                <p>Fiche 4 — en cours de développement.</p>""",
        },
        # ── ENGLISH ───────────────────────────────────────────
        "en": {
            "book_excerpt": book_html_en,
            "marine_species": extraire_especes(passage_en, "en"),
            "science_19c": f"""
                <h3 style='color:#0a3d62;'>19th-century scientific discoveries</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>🔬 {stop_en}</h5>
                <p><strong>Context:</strong> {event_en}</p>
                <p>Card 3 — under development.</p>""",
            "compare_1866_today": f"""
                <h3 style='color:#0a3d62;'>Comparison between 1866 and today</h3>
                <h5 style='color:#27c39f; font-size:0.85rem; margin-bottom:14px;'>⚗️ {stop_en}</h5>
                <p>Card 4 — under development.</p>""",
        }
    }

print(f"✅ {len(detailed_content)} escales dans detailed_content")

# Vérification rapide des 3 premières
for k in ["0", "1", "2"]:
    fr = detailed_content[k]["fr"]["book_excerpt"][:80].replace("\n","")
    en = detailed_content[k]["en"]["book_excerpt"][:80].replace("\n","")
    print(f"  [{k}] 🇫🇷 {fr}...")
    print(f"  [{k}] 🇬🇧 {en}...")

# =========================================================
# POPUP BILINGUE — exactement le même format que l'original
# =========================================================
def make_popup(row, i):
    stop_fr  = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")
    date_fr  = html_module.escape(get_lang(row.get("Date"),      "fr"))
    ocean_fr = html_module.escape(get_lang(row.get("Océan/Mer"), "fr"))
    event_fr = html_module.escape(get_lang(row.get("Event"),     "fr"))
    stop_en  = html_module.escape(get_lang(row.get("Escales du Nautilus"), "en") or f"Stop {i+1}")
    date_en  = html_module.escape(get_lang(row.get("Date"),      "en"))
    ocean_en = html_module.escape(get_lang(row.get("Océan/Mer"), "en"))
    event_en = html_module.escape(get_lang(row.get("Event"),     "en"))
    lat  = row["Latitude_Décimal"]
    lon  = row["Longitude_Décimal"]
    dist = f"{row['DistanceCentreTerre']:,}"

    return f"""
    <div style="display:flex; gap:20px; min-width:520px;">

        <!-- FRANÇAIS -->
        <div style="flex:1; border-right:1px solid #ccc; padding-right:10px;">
            <h4>Français 🇫🇷</h4>
            🗺️🧭<b>Escales du Nautilus :</b> {stop_fr}<br>
            📅 <b>Date :</b> {date_fr}<br>
            🌊 <b>Océan/Mer :</b> {ocean_fr}<br>
            📜 <b>Événement :</b> {event_fr}<br>
            ↕️ <b>Latitude :</b> {lat}<br>
            ↔️ <b>Longitude :</b> {lon}<br>
            🌍🧲 <b>Distance du centre de la Terre :</b> {dist} mètres
            <br><br>

            <div style="display:flex; gap:8px; flex-wrap:wrap;">
                <button class="submenu-toggle-btn btn btn-sm btn-primary"
                        data-menu-id="submenu-logbook-fr-{i}">
                    📖 Journal de bord vivant
                </button>
                <button class="submenu-toggle-btn btn btn-sm btn-outline-secondary"
                        data-menu-id="submenu-ai-fr-{i}">
                    ✨ Explorations IA
                </button>
            </div>

            <div id="submenu-logbook-fr-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="book_excerpt" data-group="logbook">
                        Extrait du livre
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="marine_species" data-group="logbook">
                        Espèces maritimes mentionnées
                    </button>
                </div>
            </div>

            <div id="submenu-ai-fr-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="science_19c" data-group="ai">
                        Découvertes scientifiques du XIXe siècle
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="fr" data-action="compare_1866_today" data-group="ai">
                        Comparaison entre 1866 et aujourd'hui
                    </button>
                </div>
            </div>
        </div>

        <!-- ENGLISH -->
        <div style="flex:1; padding-left:10px;">
            <h4>English 🇬🇧</h4>
            🗺️🧭<b>Nautilus Stops :</b> {stop_en}<br>
            📅 <b>Date :</b> {date_en}<br>
            🌊 <b>Ocean/Sea :</b> {ocean_en}<br>
            📜 <b>Event :</b> {event_en}<br>
            ↕️ <b>Latitude :</b> {lat}<br>
            ↔️ <b>Longitude :</b> {lon}<br>
            🌍🧲 <b>Distance from Earth's center :</b> {dist} meters
            <br><br>

            <div style="display:flex; gap:8px; flex-wrap:wrap;">
                <button class="submenu-toggle-btn btn btn-sm btn-primary"
                        data-menu-id="submenu-logbook-en-{i}">
                    📖 Living Logbook
                </button>
                <button class="submenu-toggle-btn btn btn-sm btn-outline-secondary"
                        data-menu-id="submenu-ai-en-{i}">
                    ✨ AI Explorations
                </button>
            </div>

            <div id="submenu-logbook-en-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="book_excerpt" data-group="logbook">
                        Book excerpt
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="marine_species" data-group="logbook">
                        Marine species mentioned
                    </button>
                </div>
            </div>

            <div id="submenu-ai-en-{i}" class="popup-submenu"
                 style="display:none; margin-top:10px; padding-top:6px;">
                <div style="display:flex; gap:6px; flex-wrap:wrap;">
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="science_19c" data-group="ai">
                        19th-century scientific discoveries
                    </button>
                    <button class="submenu-action btn btn-sm btn-light"
                            data-stop="{i}" data-lang="en" data-action="compare_1866_today" data-group="ai">
                        Comparison between 1866 and today
                    </button>
                </div>
            </div>
        </div>

    </div>"""

# =========================================================
# CARTE FOLIUM
# =========================================================
# =========================================================
# CARTE FOLIUM
# =========================================================
m = folium.Map(location=[20, -30], zoom_start=2, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)
coords = []

for i, row in df.iterrows():
    lat = row["Latitude_Décimal"]
    lon = row["Longitude_Décimal"]
    coords.append([lat, lon])
    tooltip = html_module.escape(get_lang(row.get("Escales du Nautilus"), "fr") or f"Escale {i+1}")

    # ── Marqueur DÉPART ──────────────────────────────────
    if i == 0:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=620),
            tooltip=f"⚓ Départ — {tooltip}",
            icon=folium.Icon(color="green", icon="anchor", prefix="fa")
        ).add_to(m)  # hors cluster pour toujours le voir

    # ── Marqueur ARRIVÉE ─────────────────────────────────
    elif i == len(df) - 1:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=620),
            tooltip=f"🏁 Arrivée — {tooltip}",
            icon=folium.Icon(color="red", icon="flag", prefix="fa")
        ).add_to(m)  # hors cluster pour toujours le voir

    # ── Marqueurs normaux ────────────────────────────────
    else:
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(make_popup(row, i), max_width=620),
            tooltip=tooltip,
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(marker_cluster)

# ── Ligne bleue du trajet ─────────────────────────────
if len(coords) > 1:
    folium.PolyLine(
        coords,
        color="darkblue",
        weight=3,
        opacity=0.75
    ).add_to(m)

print(f"✅ {len(coords)} marqueurs ajoutés — départ ⚓ vert · arrivée 🏁 rouge · trajet bleu")

# =========================================================
# INJECTION : Loader + Bootstrap Modal + JS
# Exactement le même système que ton original qui fonctionne
# =========================================================
custom_html = f"""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Michroma&display=swap" rel="stylesheet">

<!-- Loader -->
<div id="dataLoaderOverlay" style="
    display:none; position:fixed; inset:0;
    background:rgba(0,0,0,0.32); z-index:2000;
    align-items:center; justify-content:center;
    backdrop-filter:blur(2px);">
    <div style="
        padding:18px 24px; border-radius:14px;
        background:rgba(12,12,16,0.94); color:#27c39f;
        font-size:16px; font-weight:600;
        box-shadow:0 8px 24px rgba(0,0,0,0.25);
        border:1px solid rgba(39,195,159,0.22);
        font-family:Arial,sans-serif;">
        Traitement des données en cours<span class="loading-dots"></span>
    </div>
</div>

<!-- Bootstrap Modal -->
<div class="modal fade" id="logbookModal" tabindex="-1"
     aria-labelledby="logbookModalLabel" aria-hidden="true">
  <div class="modal-dialog modal-lg modal-dialog-centered">
    <div class="modal-content" style="border-radius:16px; overflow:hidden;">
      <div class="modal-header" style="background:#0a0e14; border-bottom:1px solid rgba(39,195,159,0.2);">
        <h5 class="modal-title" id="logbookModalLabel"
            style="color:#27c39f; font-family:'Michroma',sans-serif; letter-spacing:0.05em;">
            Journal de bord vivant
        </h5>
        <button type="button" class="btn-close btn-close-white"
                data-bs-dismiss="modal" aria-label="Close"></button>
      </div>
      <div class="modal-body" id="logbookModalBody"
           style="font-size:1rem; line-height:1.65; background:#0f1923; color:#c8d8e8;
                  font-family:Arial,sans-serif; padding:24px;">
      </div>
    </div>
  </div>
</div>

<style>
.leaflet-popup-content {{ min-width:520px !important; }}
.popup-submenu .btn {{ font-size:12px; }}
.loading-dots::after {{
    content:""; display:inline-block; width:1.6em; text-align:left;
    animation:dotsSteps 1.4s steps(4,end) infinite;
}}
@keyframes dotsSteps {{
    0%   {{ content:""; }}
    25%  {{ content:"."; }}
    50%  {{ content:".."; }}
    75%  {{ content:"..."; }}
    100% {{ content:""; }}
}}
</style>

<script>
const DETAILED_CONTENT = {json.dumps(detailed_content, ensure_ascii=False)};

function showLoader() {{
    const l = document.getElementById("dataLoaderOverlay");
    if (l) l.style.display = "flex";
}}
function hideLoader() {{
    const l = document.getElementById("dataLoaderOverlay");
    if (l) l.style.display = "none";
}}
function openModalWithContent(title, content) {{
    document.getElementById("logbookModalLabel").innerHTML = title;
    document.getElementById("logbookModalBody").innerHTML  = content;
    const modal = new bootstrap.Modal(document.getElementById("logbookModal"));
    modal.show();
}}

document.addEventListener("click", function(e) {{

    // Bouton toggle (Journal de bord / Explorations IA)
    const toggleBtn = e.target.closest(".submenu-toggle-btn");
    if (toggleBtn) {{
        const menuId = toggleBtn.getAttribute("data-menu-id");
        const menu   = document.getElementById(menuId);
        if (menu) {{
            menu.style.display = (menu.style.display === "none" || menu.style.display === "")
                ? "block" : "none";
        }}
        return;
    }}

    // Bouton action (Extrait du livre, Espèces, etc.)
    const actionBtn = e.target.closest(".submenu-action");
    if (actionBtn) {{
        const stop   = actionBtn.getAttribute("data-stop");
        const lang   = actionBtn.getAttribute("data-lang") || "fr";
        const action = actionBtn.getAttribute("data-action");
        const group  = actionBtn.getAttribute("data-group");

        const titles = {{
            fr: {{
                book_excerpt:       "📖 Extrait du livre",
                marine_species:     "🐙 Espèces maritimes mentionnées",
                science_19c:        "🔬 Découvertes scientifiques du XIXe siècle",
                compare_1866_today: "⚗️ Comparaison entre 1866 et aujourd'hui"
            }},
            en: {{
                book_excerpt:       "📖 Book excerpt",
                marine_species:     "🐙 Marine species mentioned",
                science_19c:        "🔬 19th-century scientific discoveries",
                compare_1866_today: "⚗️ Comparison between 1866 and today"
            }}
        }};

        const fallback = lang === "fr"
            ? "<p>Contenu disponible bientôt.</p>"
            : "<p>Content coming soon.</p>";

        const content = (
            DETAILED_CONTENT[stop] &&
            DETAILED_CONTENT[stop][lang] &&
            DETAILED_CONTENT[stop][lang][action]
        ) ? DETAILED_CONTENT[stop][lang][action] : fallback;

        if (group === "ai") {{
            showLoader();
            setTimeout(() => {{
                hideLoader();
                openModalWithContent(titles[lang][action] || "Exploration IA", content);
            }}, 2400);
        }} else {{
            openModalWithContent(titles[lang][action] || "Journal de bord vivant", content);
        }}
        return;
    }}
}});
</script>
"""

m.get_root().html.add_child(folium.Element(custom_html))

fix_map = """
<script>
window.addEventListener('load', function() {
    setTimeout(function() {
        var mapId = Object.keys(window).filter(k => k.startsWith('map_'))[0];
        if (window[mapId]) {
            window[mapId].invalidateSize();
            window[mapId].setView([20, -30], 2);
        }
    }, 300);
});
</script>
"""
m.get_root().html.add_child(folium.Element(fix_map))

# =========================================================
# EXPORT
# =========================================================
m.save(str(OUTPUT_HTML))
print(f"✅ Fichier généré : {OUTPUT_HTML.resolve()}")
print()
print("📋 Prochaine étape :")
print(f"   → Copie '{OUTPUT_HTML.name}' dans ton projet VSCode")
print(f"   → Remplace 'carte_nautilus_multilang14.html'")
print(f"   → Deploy sur Netlify")

display(HTML(f'<a href="{OUTPUT_HTML.name}" target="_blank">🔗 Ouvrir la carte HTML générée</a>')) 





✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\data_texts
✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\data_texts\20000lieues_fr.txt
✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\data_texts\20000lieues_an.txt
✅ C:\Users\Utilisateur\Documents\Nautilus_IA_Visualisation\notebooks\escales_enrichies_dates_corrigees.json
📖 FR : 908,396 caractères
📖 EN : 617,239 caractères
🗺️  32 escales chargées
✅ 32 escales avec coordonnées valides
✅ Fiche 2 — 86 espèces FR · 47 espèces EN dans le dictionnaire
✅ 32 escales dans detailed_content
  [0] 🇫🇷         <h3 style='color:#0a3d62; font-size:1.3rem; margin-bottom:8px;'>Île de ...
  [0] 🇬🇧         <h3 style='color:#0a3d62; font-size:1.3rem; margin-bottom:8px;'>Crespo ...
  [1] 🇫🇷         <h3 style='color:#0a3d62; font-size:1.3rem; margin-bottom:8px;'>Archipe...
  [1] 🇬🇧         <h3 style='color:#0a3d62; font-size:1.3rem; margin-bottom:8px;'>Pomotou...
  [2] 🇫🇷         <h3 style='color: